# topology_metrics usage

This notebook records how to build and read full-link shortest-path edge usage share stores. Each code cell is self-contained so it can be run independently.

## Build G60 full-link usage share

This runs the reusable example script for the G60 paper1 three region pairs. It writes both weighted shortest-delay usage share and unweighted shortest-hop usage share.

In [ ]:
import subprocess
from pathlib import Path

PYTHON = Path(r"C:\ProgramData\miniconda3\envs\paper11\python.exe")
SCRIPT = Path(r"E:\paper11\generic\src\topology_metrics\examples\run_g60_full_link_edge_usage_three_pairs.py")
OUT_DIR = Path(r"E:\paper11\data\satnet_experiments\runs\paper1\G60\full_link_edge_usage_share_three_pairs_t0_86160_stride60")

cmd = [
    str(PYTHON),
    str(SCRIPT),
    "--start", "0",
    "--end", "86160",
    "--stride", "60",
    "--workers", "8",
    "--progress-every", "100",
    "--out-dir", str(OUT_DIR),
]
subprocess.run(cmd, check=True)
OUT_DIR

## Read critical edge vectors

`combined_usage_share_max_over_time.npy` is aligned with `edges.csv`. These vectors can be used as transition weights in topology selectors.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

STORE = Path(r"E:\paper11\data\satnet_experiments\runs\paper1\G60\full_link_edge_usage_share_three_pairs_t0_86160_stride60")
delay_edges = pd.read_csv(STORE / "delay_shortest" / "edges.csv")
delay_share = np.load(STORE / "delay_shortest" / "combined_usage_share_max_over_time.npy")
hop_share = np.load(STORE / "hop_shortest" / "combined_usage_share_max_over_time.npy")

print("edges", delay_edges.shape)
print("delay share", delay_share.shape, float(delay_share.max()), float(delay_share.mean()))
print("hop share", hop_share.shape, float(hop_share.max()), float(hop_share.mean()))
delay_edges.assign(delay_usage_share=delay_share, hop_usage_share=hop_share).sort_values(
    "delay_usage_share", ascending=False
).head(10)

## Use usage share in topology_learning selector

The selector config below adds `critical_edge_transition_profiles`. The module still keeps region groups as metric endpoint sets only; it does not impose region-internal `+grid` links.

In [ ]:
import subprocess
from pathlib import Path

PYTHON = Path(r"C:\ProgramData\miniconda3\envs\paper11\python.exe")
CONFIG = Path(r"E:\paper11\generic\src\topology_learning\examples\configs\g60_w4h3_full_link_gap_three_pairs_hop_delay_critical_edges.yaml")
SCRIPT = Path(r"E:\paper11\generic\src\topology_learning\examples\run_full_link_gap_selector.py")

subprocess.run([str(PYTHON), str(SCRIPT), "--config", str(CONFIG)], check=True)
CONFIG